In [8]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [9]:
df = pd.read_csv("tissue_dataset.csv")
X = df.drop(columns = ["target", "target_actual"])
y = df["target"]
top25_miRNA = pd.read_csv("results/optimal_25_miRNA_panel.csv")
selected_features = top25_miRNA["miRNA"].tolist()
X_optimal = X[selected_features]
X_optimal.shape

(133, 25)

In [10]:
#Classification using SVM
pipeline = Pipeline([
    (
        "smote",
        SMOTE(random_state = 42)
    ),
    (
        "svm",
        SVC(
            kernel = "linear",
            probability = True,
            random_state = 42
        )
    )
])
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)
scores = cross_validate(
    pipeline,
    X_optimal,
    y,
    cv = cv,
     scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }
)

validation_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Mean": [
        scores["test_accuracy"].mean(),
        scores["test_precision"].mean(),
        scores["test_recall"].mean(),
        scores["test_f1"].mean(),
        scores["test_roc_auc"].mean()
    ],
    "Std": [
        scores["test_accuracy"].std(),
        scores["test_precision"].std(),
        scores["test_recall"].std(),
        scores["test_f1"].std(),
        scores["test_roc_auc"].std()
    ]
})
validation_results

,Metric,Mean,Std
0,Accuracy,0.992308,0.015385
1,Precision,0.992000,0.016000
2,Recall,1.000000,0.000000
3,F1 Score,0.995918,0.008163
4,ROC-AUC,0.991667,0.016667
